# Notebook 03 — Modelos Baseline

## Objetivo

Entrenar modelos clásicos de clasificación supervisada sobre representaciones BoW y TF-IDF.

**Experimento principal:** subconjunto político.  
**Experimento complementario:** dataset completo (solo baselines).

> El modelo aprende patrones lingüísticos del dataset; no verifica hechos.

**Métrica principal:** F2-score de la clase fake (prioriza recall de fake sin ignorar precisión).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlp.paths import *
from nlp.plotting import setup_style, save_figure

setup_style()

from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, roc_curve, auc
import joblib

from nlp.io import load_splits
from nlp.metrics import compute_metrics, metrics_to_row
from nlp.modeling import (
    baseline_text_columns,
    build_model,
    build_vectorizer,
    get_text_col,
    predict_proba_scores,
    run_baseline_grid,
)
from nlp.plotting import plot_confusion_matrix


## 1. Carga de splits

In [2]:
BASELINE_COLS = baseline_text_columns()
pol_train, pol_val, pol_test = load_splits('politics', columns=BASELINE_COLS)
full_train, full_val, full_test = load_splits('full', columns=BASELINE_COLS)


## 2. Configuración de la grilla de experimentos

In [3]:

NGRAM_OPTS = [(1, 1), (1, 2)]
MAX_FEATURES_OPTS = [10000, 30000, 50000]
VECTORIZER_TYPES = ['bow', 'tfidf']


## 3. Entrenamiento y selección por validación (F2 fake)

In [4]:

def build_pipeline_from_config(config_row):
    ngram = eval(config_row['ngram_range'])
    return Pipeline([
        ('vec', build_vectorizer(
            config_row['vectorizer'], ngram, int(config_row['max_features'])
        )),
        ('clf', build_model(config_row['model'], config_row['best_param'])),
    ])


def evaluate_on_test(config_row, train, test):
    """Entrena con train y evalúa una sola vez en test."""
    text_col = get_text_col(config_row['text_field'], config_row['stopwords'])
    pipe = build_pipeline_from_config(config_row)
    pipe.fit(train[text_col].fillna(''), train['label'])

    X_test = test[text_col].fillna('')
    y_test_pred = pipe.predict(X_test)
    y_proba = predict_proba_scores(pipe, X_test)

    test_m = compute_metrics(test['label'], y_test_pred, y_proba)
    result_row = metrics_to_row(test_m, {
        'model': config_row['model'],
        'vectorizer': config_row['vectorizer'],
        'text_field': config_row['text_field'],
        'stopwords': config_row['stopwords'],
        'ngram_range': config_row['ngram_range'],
        'max_features': int(config_row['max_features']),
        'best_param': config_row['best_param'],
        'dataset_scope': config_row['dataset_scope'],
        'split': 'test',
    })
    return pipe, result_row

grid_kwargs = dict(
    ngram_opts=NGRAM_OPTS,
    max_features_opts=MAX_FEATURES_OPTS,
    vectorizer_types=VECTORIZER_TYPES,
)
if DEV_MODE:
    grid_kwargs['max_combos'] = 20
    print('DEV_MODE activo: grilla politics limitada a max_combos=20')

# Experimento principal: subconjunto político (solo métricas de validación)
politics_results = run_baseline_grid(pol_train, pol_val, 'politics', **grid_kwargs)


DEV_MODE activo: grilla politics limitada a max_combos=20


politics:   0%|          | 0/72 [00:00<?, ?it/s]

In [5]:
if DEV_MODE:
    print('DEV_MODE activo: se omite la grilla full_dataset')
    full_results = pd.DataFrame()
else:
    full_results = run_baseline_grid(
        full_train, full_val, 'full_dataset',
        ngram_opts=NGRAM_OPTS,
        max_features_opts=MAX_FEATURES_OPTS,
        vectorizer_types=VECTORIZER_TYPES,
    )

val_results = politics_results if full_results.empty else pd.concat(
    [politics_results, full_results], ignore_index=True,
)
print('Mejores 10 configuraciones según validación:')
print(val_results.sort_values('f2_fake', ascending=False).head(10).to_string(index=False))


DEV_MODE activo: se omite la grilla full_dataset
Mejores 10 configuraciones según validación:
 accuracy  precision_fake  recall_fake  f1_fake  f2_fake  roc_auc               model vectorizer text_field      stopwords ngram_range  max_features  best_param dataset_scope split
 0.957518        0.911184     0.960694 0.935284 0.950366 0.990315      multinomial_nb        bow      title with_stopwords      (1, 2)         10000         1.0      politics   val
 0.951607        0.897186     0.958382 0.926775 0.945484 0.990818      multinomial_nb        bow      title with_stopwords      (1, 1)         30000         0.1      politics   val
 0.951607        0.897186     0.958382 0.926775 0.945484 0.990818      multinomial_nb        bow      title with_stopwords      (1, 1)         50000         0.1      politics   val
 0.951607        0.897186     0.958382 0.926775 0.945484 0.990818      multinomial_nb        bow      title with_stopwords      (1, 1)         10000         0.1      politics   val
 

## 4. Mejor modelo (selección en val) y evaluación final en test

In [6]:

# Selección final por validación; test solo para el mejor modelo de cada alcance
pol_best_val = politics_results.sort_values('f2_fake', ascending=False).iloc[0]
pipe_pol, pol_best_test = evaluate_on_test(pol_best_val, pol_train, pol_test)

test_rows = [pol_best_test]
if not full_results.empty:
    full_best_val = full_results.sort_values('f2_fake', ascending=False).iloc[0]
    _, full_best_test = evaluate_on_test(full_best_val, full_train, full_test)
    test_rows.append(full_best_test)

baseline_results = pd.concat([val_results, pd.DataFrame(test_rows)], ignore_index=True)
baseline_results.to_csv(RESULTS_METRICS / 'baseline_results.csv', index=False)

print('Mejor configuración politics (val):')
print(pol_best_val.to_string())
print('\nEvaluación en test del mejor modelo politics:')
print(pd.Series(pol_best_test).to_string())

text_col = get_text_col(pol_best_test['text_field'], pol_best_test['stopwords'])
y_pred = pipe_pol.predict(pol_test[text_col].fillna(''))
cm = confusion_matrix(pol_test['label'], y_pred)

plot_confusion_matrix(
    cm, ['Real', 'Fake'],
    f'Matriz de confusión — {pol_best_test["model"]}',
    RESULTS_FIGURES / 'baseline_best_confusion_matrix.png',
)

X_test = pol_test[text_col].fillna('')
y_proba = predict_proba_scores(pipe_pol, X_test)

if y_proba is not None:
    fpr, tpr, _ = roc_curve(pol_test['label'], y_proba)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('Curva ROC — mejor baseline (politics, test)')
    ax.legend()
    save_figure(fig, RESULTS_FIGURES / 'baseline_best_roc_curve.png')
    plt.show()

joblib.dump(pipe_pol, RESULTS_MODELS / 'best_baseline_politics.joblib')
pd.Series(pol_best_test).to_json(RESULTS_MODELS / 'best_baseline_politics_config.json')
joblib.dump(pipe_pol, RESULTS_MODELS / 'best_baseline_model.joblib')
print('Modelos guardados en results/models/')


Mejor configuración politics (val):
accuracy                0.957518
precision_fake          0.911184
recall_fake             0.960694
f1_fake                 0.935284
f2_fake                 0.950366
roc_auc                 0.990315
model             multinomial_nb
vectorizer                   bow
text_field                 title
stopwords         with_stopwords
ngram_range               (1, 2)
max_features               10000
best_param                   1.0
dataset_scope           politics
split                        val

Evaluación en test del mejor modelo politics:
accuracy                0.962334
precision_fake          0.942478
recall_fake             0.944568
f1_fake                 0.943522
f2_fake                 0.944149
roc_auc                  0.98743
model             multinomial_nb
vectorizer                   bow
text_field                 title
stopwords         with_stopwords
ngram_range               (1, 2)
max_features               10000
best_param                

## 5. Comparación politics vs full_dataset

Si el rendimiento en el dataset completo es mucho mayor que en el subconjunto político, probablemente parte del rendimiento se explica por sesgos de tema, fuente o estructura del dataset — no solo por patrones lingüísticos de fake news.

In [7]:

compare_val = val_results.groupby('dataset_scope')['f2_fake'].max()
compare_test = baseline_results[baseline_results['split'] == 'test'].set_index('dataset_scope')['f2_fake']
print('Mejor F2 en validación por alcance:')
print(compare_val)
print('\nF2 en test del mejor modelo seleccionado en val:')
print(compare_test)

fig, ax = plt.subplots(figsize=(8, 4))
top_pol = politics_results.nlargest(5, 'f2_fake')['f2_fake'].mean()
if full_results.empty:
    sns.barplot(x=['politics (top-5 avg)'], y=[top_pol], ax=ax)
else:
    top_full = full_results.nlargest(5, 'f2_fake')['f2_fake'].mean()
    sns.barplot(
        x=['politics (top-5 avg)', 'full_dataset (top-5 avg)'],
        y=[top_pol, top_full],
        ax=ax,
    )
ax.set_ylabel('F2 fake en validación (promedio top-5)')
ax.set_title('Comparación subconjunto político vs dataset completo (val)')
save_figure(fig, RESULTS_FIGURES / 'baseline_politics_vs_full.png')
plt.show()


Mejor F2 en validación por alcance:
dataset_scope
politics    0.950366
Name: f2_fake, dtype: float64

F2 en test del mejor modelo seleccionado en val:
dataset_scope
politics    0.944149
Name: f2_fake, dtype: float64


## Conclusiones

- Se compararon LR, MNB y Linear SVM con BoW y TF-IDF.
- Se evaluaron título, cuerpo y título+cuerpo; con y sin stopwords.
- La métrica principal F2 prioriza detectar fake news (minimizar falsos negativos).
- El modelo aprende patrones del dataset, no verifica hechos.